In [3]:
# Installation rapide
!pip install polars s3fs boto3 pyarrow pandas openpyxl --quiet


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import boto3
import pandas as pd
import numpy as np
from io import BytesIO
from io import StringIO
import re
import io
import os

In [5]:
s3_client = boto3.client(
 "s3",
 endpoint_url="http://minio.mon-namespace.svc.cluster.local:80", # service interne K8s
 aws_access_key_id="Aminata",
 aws_secret_access_key="AminataCae312",
 verify=False # à désactiver si SSL auto-signé ; sinon mettre True avec certificat valide
)

In [6]:
# Paramètres
bucket_name = "admindataanstat"
object_key = "Solde/panel_solde_df.csv"  # Chemin dans le bucket

In [7]:
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)
panel_solde_df = pd.read_csv(io.BytesIO(obj["Body"].read()), encoding="utf-8-sig")


In [8]:
# Affiche les 5 premières lignes pour un aperçu rapide.
panel_solde_df.head()

,MATRICULE||CODE_ORGANISME,MATRICULE,NOM,DATE NAISSANCE,SEXE,SITUATION MATRIMONIALE,NOMBRE ENFANT,LIEU AFFECTATION,SERVICE,ORGANISME,...,MONTANT NET,RETENUE PENSION,IMPOT,CHARGE PATRONALE,MOIS/ANNEE,GRADE 2,AGE RETRAITE,DATE RETRAITE,PERIODE,DATE_COLLECTE
0,011242X15,011242X,DOSSO MEGBO,1939-01-01 00:00:00,Masculin,Marié,0.0,Seguela,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,34145,0.0,415.0,0.0,12015,60,NaN,NaN,12015,2015-01-01
1,012856Q15,012856Q,KHISSY BEYNIOUAH FULBERT,1939-01-01 00:00:00,Masculin,Célibataire,0.0,Bouaké,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,41582,0.0,509.0,0.0,12015,60,NaN,NaN,12015,2015-01-01
2,013454N15,013454N,VAHA TOMAS GNOMBLEI HENRI,1924-01-01 00:00:00,Masculin,Marié,0.0,Guiglo,D. G. A. A. T.,Min. d'Etat Administration du Territoire,...,45053,0.0,547.0,0.0,12015,60,NaN,NaN,12015,2015-01-01
3,027861L25,027861L,CAPET YATO MATHIEU,1943-03-01 00:00:00,Masculin,Marié,0.0,Abidjan,A affecter,"Min. Affaires Etrangères, de l'Intégration Afr...",...,662121,0.0,9600.0,0.0,12015,60,NaN,NaN,12015,2015-01-01
4,039659M38,039659M,COULIBALY YOUSSOUF,1945-12-01 00:00:00,Masculin,Marié,2.0,Abidjan,Dir Personnel Police Nationale,Min. de l'Intérieur et de la Sécurité,...,810000,0.0,0.0,0.0,12015,A7,NaN,NaN,12015,2015-01-01


In [9]:
# Afficher la liste des variable de ma base de données
print(panel_solde_df.columns)

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE'],
      dtype='object')


In [ ]:
#################################################################
#                   MATRICULE||CODE_ORGANISME                   #
#################################################################

In [10]:
# Examen de la variable MATRICULE||CODE_ORGANISME
panel_solde_df["MATRICULE||CODE_ORGANISME"].describe()

count      15627963
unique       308240
top       612155S15
freq             72
Name: MATRICULE||CODE_ORGANISME, dtype: object

In [11]:
# Créer une nouvelle colonne qui compte le nombre de position de la variable MATRICULE
panel_solde_df["nb_digits_mat"] = panel_solde_df["MATRICULE||CODE_ORGANISME"].str.len()
print(panel_solde_df["nb_digits_mat"])

0           9
1           9
2           9
3           9
4           9
           ..
15627958    9
15627959    9
15627960    9
15627961    9
15627962    9
Name: nb_digits_mat, Length: 15627963, dtype: int64


In [12]:
# Filtrer les lignes où le code ne tient pas sur 9 chiffres
codes_invalides = panel_solde_df[panel_solde_df["nb_digits_mat"] != 9]
print(codes_invalides[["MATRICULE||CODE_ORGANISME", "nb_digits_mat"]])  
# Tous les individus de notre base ont un matricule et code organisme qui tient sur 9 positions (Matricule + code organisme).

Empty DataFrame
Columns: [MATRICULE||CODE_ORGANISME, nb_digits_mat]
Index: []


In [13]:
# Suppression de la variable nombre de digit de la variable MATRICULE||CODE_ORGANISME
panel_solde_df = panel_solde_df.drop(columns=["nb_digits_mat"])

In [14]:
# Extraire les 7 premiers caractères pour le MATRICULE
panel_solde_df["MATRICULE_Verif"] = panel_solde_df["MATRICULE||CODE_ORGANISME"].str[:7]

# Extraire les caractères à partir de la position 7 pour le CODE_ORGANISME
panel_solde_df["CODE_ORGANISME"] = panel_solde_df["MATRICULE||CODE_ORGANISME"].str[7:]

In [15]:
panel_solde_df["MATRICULE_Verif"].str.len()

0           7
1           7
2           7
3           7
4           7
           ..
15627958    7
15627959    7
15627960    7
15627961    7
15627962    7
Name: MATRICULE_Verif, Length: 15627963, dtype: int64

In [16]:
panel_solde_df["CODE_ORGANISME"].str.len()

0           2
1           2
2           2
3           2
4           2
           ..
15627958    2
15627959    2
15627960    2
15627961    2
15627962    2
Name: CODE_ORGANISME, Length: 15627963, dtype: int64

In [17]:
# Filtrer les lignes où le matricule obtenu est différent du matricule fourmi dans la base de données 
mat_diff = panel_solde_df[panel_solde_df["MATRICULE_Verif"] != panel_solde_df["MATRICULE"]]
mat_diff

,MATRICULE||CODE_ORGANISME,MATRICULE,NOM,DATE NAISSANCE,SEXE,SITUATION MATRIMONIALE,NOMBRE ENFANT,LIEU AFFECTATION,SERVICE,ORGANISME,...,IMPOT,CHARGE PATRONALE,MOIS/ANNEE,GRADE 2,AGE RETRAITE,DATE RETRAITE,PERIODE,DATE_COLLECTE,MATRICULE_Verif,CODE_ORGANISME


In [18]:
# Suppression de la variable nombre de digit de la variable MATRICULE VERIF
panel_solde_df = panel_solde_df.drop(columns=["MATRICULE_Verif"])
print(panel_solde_df.columns)

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'CODE_ORGANISME'],
      dtype='object')


In [71]:
#################################################################
#                            DATE NAISSANCE                     #
#################################################################

In [19]:
# Examen de la variable Date de naissance
panel_solde_df["DATE NAISSANCE"].describe()

count                15620485
unique                  15455
top       1980-01-01 00:00:00
freq                   176217
Name: DATE NAISSANCE, dtype: object

In [20]:
# Conversion au format datetime
panel_solde_df["DATE NAISSANCE"] = pd.to_datetime(panel_solde_df["DATE NAISSANCE"], errors='coerce')

# Extraire l'année, le mois et le jour
panel_solde_df["ANNEE_NAISSANCE"] = panel_solde_df["DATE NAISSANCE"].dt.year
panel_solde_df["MOIS_NAISSANCE"] = panel_solde_df["DATE NAISSANCE"].dt.month
panel_solde_df["JOUR_NAISSANCE"] = panel_solde_df["DATE NAISSANCE"].dt.day

In [21]:
# # Création du tableau avec effectifs et pourcentages (y compris NaN)
tab_annee = panel_solde_df["ANNEE_NAISSANCE"].value_counts(dropna=False).sort_index()
tab_annee_pct = panel_solde_df["ANNEE_NAISSANCE"].value_counts(normalize=True, dropna=False).sort_index()

# Créer le DataFrame et l’assigner à une variable
df_tab = pd.DataFrame({
    "Effectif": tab_annee,
    "Pourcentage (%)": (tab_annee_pct * 100).round(2)
})
df_tab

,Effectif,Pourcentage (%)
ANNEE_NAISSANCE,,
1900.0,11,0.00
1924.0,60,0.00
1931.0,4,0.00
1933.0,6,0.00
1935.0,60,0.00
...,...,...
1999.0,443,0.00
2000.0,182,0.00
2011.0,17009,0.11


In [22]:
# # Création du tableau avec effectifs et pourcentages (y compris NaN)
tab_mois = panel_solde_df["MOIS_NAISSANCE"].value_counts(dropna=False).sort_index()
tab_mois_pct = panel_solde_df["MOIS_NAISSANCE"].value_counts(normalize=True, dropna=False).sort_index()

# Créer le DataFrame et l’assigner à une variable
pd.DataFrame({
    "Effectif": tab_mois,
    "Pourcentage (%)": (tab_mois_pct * 100).round(2)
})

,Effectif,Pourcentage (%)
MOIS_NAISSANCE,,
1.0,2881377,18.44
2.0,926638,5.93
3.0,1114726,7.13
4.0,1059270,6.78
5.0,1098128,7.03
6.0,1099144,7.03
7.0,1009007,6.46
8.0,937539,6.00
9.0,909392,5.82


In [23]:
# # Création du tableau avec effectifs et pourcentages (y compris NaN)
tab_jour = panel_solde_df["JOUR_NAISSANCE"].value_counts(dropna=False).sort_index()
tab_jour_pct = panel_solde_df["JOUR_NAISSANCE"].value_counts(normalize=True, dropna=False).sort_index()

# Créer le DataFrame et l’assigner à une variable
pd.DataFrame({
    "Effectif": tab_jour,
    "Pourcentage (%)": (tab_jour_pct * 100).round(2)
})

,Effectif,Pourcentage (%)
JOUR_NAISSANCE,,
1.0,5288630,33.84
2.0,354030,2.27
3.0,309203,1.98
4.0,304176,1.95
5.0,346069,2.21
6.0,290310,1.86
7.0,287207,1.84
8.0,301451,1.93
9.0,277218,1.77


In [24]:
# Création du buffer mémoire binaire pour Excel
excel_buffer = BytesIO()

# Écriture du DataFrame dans ce buffer
with pd.ExcelWriter(excel_buffer, engine='openpyxl') as writer:
        df_tab.to_excel(writer, index=True, sheet_name='Tab_AnneeNaissance')

# Remise à zéro du buffer pour lecture complète
excel_buffer.seek(0)

# Upload vers MinIO S3
s3_client.put_object(
    Bucket="admindataanstat",  # ton bucket
    Key="Solde/tabulation_annee_naissance.xlsx",  # chemin/fichier dans le bucket
    Body=excel_buffer.getvalue()
)

{'ResponseMetadata': {'RequestId': '185311DE8FAFC1D9',
  'HostId': '59522c47b885b9de00b5168596b7b7d388e67f85c178eb9e65eb7866b433cb05',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'accept-ranges': 'bytes',
   'content-length': '0',
   'etag': '"e955039fa37d9f8bbfbf2515ae772154"',
   'server': 'MinIO',
   'strict-transport-security': 'max-age=31536000; includeSubDomains',
   'vary': 'Origin, Accept-Encoding',
   'x-amz-checksum-crc32': '5Iuk8w==',
   'x-amz-id-2': '59522c47b885b9de00b5168596b7b7d388e67f85c178eb9e65eb7866b433cb05',
   'x-amz-request-id': '185311DE8FAFC1D9',
   'x-content-type-options': 'nosniff',
   'x-ratelimit-limit': '99089',
   'x-ratelimit-remaining': '99089',
   'x-xss-protection': '1; mode=block',
   'x-amz-version-id': '268c2ac9-b2ef-41ff-b4dc-d74a31be84ce',
   'date': 'Thu, 17 Jul 2025 14:58:47 GMT'},
  'RetryAttempts': 0},
 'ETag': '"e955039fa37d9f8bbfbf2515ae772154"',
 'ChecksumCRC32': '5Iuk8w==',
 'VersionId': '268c2ac9-b2ef-41ff-b4dc-d74a31be84ce'}

In [72]:
#################################################################
#                            DATE COLLECTE                      #
#################################################################

In [43]:
# S'assurer que DATE_COLLECTE est bien au format datetime
panel_solde_df["DATE_COLLECTE"] = pd.to_datetime(panel_solde_df["DATE_COLLECTE"], errors='coerce')

In [26]:
# Conversion en numérique (float), les erreurs sont mises à NaN
panel_solde_df["ANNEE_COLLECTE"] = pd.to_numeric(panel_solde_df["ANNEE_COLLECTE"], errors="coerce")

# Extraire l'année, le mois et le jour
panel_solde_df["ANNEE_COLLECTE"] = panel_solde_df["DATE_COLLECTE"].dt.year
panel_solde_df["MOIS_COLLECTE"] = panel_solde_df["DATE_COLLECTE"].dt.month
panel_solde_df["JOUR_COLLECTE"] = panel_solde_df["DATE_COLLECTE"].dt.day

In [27]:
# Tabulation de la variable année de collecte
tab_annee_collecte = panel_solde_df["ANNEE_COLLECTE"].value_counts(dropna=False).sort_index()
tab_annee_collecte_pct = panel_solde_df["ANNEE_COLLECTE"].value_counts(normalize=True, dropna=False).sort_index()

pd.DataFrame({
    "Effectif": tab_annee_collecte,
    "Pourcentage (%)": (tab_annee_collecte_pct * 100).round(2)
})

,Effectif,Pourcentage (%)
ANNEE_COLLECTE,,
2015,2304670,14.75
2016,2427817,15.54
2017,2509596,16.06
2018,2668828,17.08
2019,2795579,17.89
2020,2921473,18.69


In [28]:
# Tabulation de la variable mois de collecte
tab_mois_collecte = panel_solde_df["MOIS_COLLECTE"].value_counts(dropna=False).sort_index()
tab_mois_collecte_pct = panel_solde_df["MOIS_COLLECTE"].value_counts(normalize=True, dropna=False).sort_index()

pd.DataFrame({
    "Effectif": tab_mois_collecte,
    "Pourcentage (%)": (tab_mois_collecte_pct * 100).round(2)
})

,Effectif,Pourcentage (%)
MOIS_COLLECTE,,
1,1246407,7.98
2,1260660,8.07
3,1269924,8.13
4,1281866,8.20
5,1289817,8.25
6,1302055,8.33
7,1310055,8.38
8,1319467,8.44
9,1327515,8.49


In [29]:
# Tabulation de la variable mois de collecte
tab_jour_collecte = panel_solde_df["JOUR_COLLECTE"].value_counts(dropna=False).sort_index()
tab_jour_collecte_pct = panel_solde_df["JOUR_COLLECTE"].value_counts(normalize=True, dropna=False).sort_index()

pd.DataFrame({
    "Effectif": tab_jour_collecte,
    "Pourcentage (%)": (tab_jour_collecte_pct * 100).round(2)
})

,Effectif,Pourcentage (%)
JOUR_COLLECTE,,
1,15627963,100.0


In [74]:
#################################################################
#          METHODE DE MISE A JOUR DE LA DATE DE NAISSANCE       #
#################################################################

In [34]:
# J'ai des données de panel mais je cherche à savoir si pour un individu j'ai plusieurs date de naissance en fonction de la date de collecte. Il peut mettre à jour ces informations

# Regrouper par identifiant individuel et compter le nombre de dates de naissance uniques
verif_naissance = panel_solde_df.groupby("MATRICULE")["DATE NAISSANCE"].nunique()

# Filtrer ceux ayant plus d'une date de naissance
individus_naissances_diff = verif_naissance[verif_naissance > 1]

# Afficher les identifiants concernés
individus_naissances_diff.describe().round(0)

count    50543.0
mean         2.0
std          0.0
min          2.0
25%          2.0
50%          2.0
75%          2.0
max          5.0
Name: DATE NAISSANCE, dtype: float64

In [39]:
# Calculer le nombre de dates de naissance différentes par individu
nbre_date_naiss = panel_solde_df.groupby("MATRICULE")["DATE NAISSANCE"].nunique()

# Ajouter cette information au DataFrame principal
panel_solde_df["nbre_date_naissances"] = panel_solde_df["MATRICULE"].map(nbre_date_naiss)

In [40]:
# Tabulation de la variable mois de collecte
tab_ind_nais = panel_solde_df["nbre_date_naissances"].value_counts(dropna=False).sort_index()
tab_ind_nais_pct = panel_solde_df["nbre_date_naissances"].value_counts(normalize=True, dropna=False).sort_index()

pd.DataFrame({
    "Effectif": tab_ind_nais,
    "Pourcentage (%)": (tab_ind_nais_pct * 100).round(2)
})

,Effectif,Pourcentage (%)
nbre_date_naissances,,
0,211,0.00
1,11942618,76.42
2,3443053,22.03
3,226139,1.45
4,15568,0.10
5,374,0.00


In [48]:
# Trier par MATRICULE et DATE_COLLECTE croissante
panel_sorted = panel_solde_df.sort_values(by=["MATRICULE", "DATE_COLLECTE"], ascending=[True, False])

# Garder uniquement la première ligne pour chaque MATRICULE (donc la collecte la plus récente)
date_naiss_corrigee = panel_sorted.groupby("MATRICULE").first().reset_index()

# Garder seulement MATRICULE et DATE NAISSANCE
date_naiss_corrigee = date_naiss_corrigee[["MATRICULE", "DATE NAISSANCE"]]

# Fusionner avec le DataFrame original pour créer une colonne "DATE NAISSANCE CORRIGEE"
panel_solde_df["DATE_NAISSANCE_CORRIGEE"] = panel_solde_df["MATRICULE"].map(
    date_naiss_corrigee.set_index("MATRICULE")["DATE NAISSANCE"]
)

In [50]:
# S'assurer que les colonnes de dates sont bien en format datetime
panel_solde_df["DATE_NAISSANCE_CORRIGEE"] = pd.to_datetime(panel_solde_df["DATE_NAISSANCE_CORRIGEE"])

# Extraire l'année, le mois et le jour
panel_solde_df["ANNEE_NAISSANCE_CORRIGEE"] = panel_solde_df["DATE_NAISSANCE_CORRIGEE"].dt.year
panel_solde_df["MOIS_NAISSANCE_CORRIGEE"] = panel_solde_df["DATE_NAISSANCE_CORRIGEE"].dt.month
panel_solde_df["JOUR_NAISSANCE_CORRIGEE"] = panel_solde_df["DATE_NAISSANCE_CORRIGEE"].dt.day

In [51]:
panel_solde_df["ANNEE_NAISSANCE_CORRIGEE"].describe().round(0)

count    15627752.0
mean         1975.0
std             9.0
min          1900.0
25%          1968.0
50%          1976.0
75%          1982.0
max          2011.0
Name: ANNEE_NAISSANCE_CORRIGEE, dtype: float64

In [79]:
#################################################################
#                             AGE                               #
#################################################################

In [52]:
# Tabulation de la variable année de naissance corrigé
tab_annee_naiss_cor = panel_solde_df["ANNEE_NAISSANCE_CORRIGEE"].value_counts(dropna=False).sort_index()
tab_annee_naiss_cor_pct = panel_solde_df["ANNEE_NAISSANCE_CORRIGEE"].value_counts(normalize=True, dropna=False).sort_index()

pd.DataFrame({
    "Effectif": tab_annee_naiss_cor,
    "Pourcentage (%)": (tab_annee_naiss_cor_pct * 100).round(2)
})

,Effectif,Pourcentage (%)
ANNEE_NAISSANCE_CORRIGEE,,
1900.0,11,0.00
1924.0,60,0.00
1931.0,4,0.00
1933.0,6,0.00
1935.0,60,0.00
...,...,...
1998.0,1971,0.01
1999.0,443,0.00
2000.0,182,0.00


In [58]:
# Calculer l'âge exact du fonctionnaire
def calcul_age_si_date(row):
    if pd.notna(row["DATE_NAISSANCE_CORRIGEE"]):
        age = row["DATE_COLLECTE"].year - row["DATE_NAISSANCE_CORRIGEE"].year - (
            (row["DATE_COLLECTE"].month, row["DATE_COLLECTE"].day) < (row["DATE_NAISSANCE_CORRIGEE"].month, row["DATE_NAISSANCE_CORRIGEE"].day)
        )
        return age
    else:
        return pd.NA  
        
panel_solde_df["AGE"] = panel_solde_df.apply(calcul_age_si_date, axis=1)

In [76]:
# Tabulation de la variable age
age = panel_solde_df["AGE"].value_counts(dropna=False).sort_index()
age_pct = panel_solde_df["AGE"].value_counts(normalize=True, dropna=False).sort_index()

pd.DataFrame({
    "Effectif": age,
    "Pourcentage (%)": (age_pct * 100).round(2)

})

,Effectif,Pourcentage (%)
AGE,,
3.0,80,0.0
4.0,104,0.0
5.0,72,0.0
6.0,52,0.0
7.0,40,0.0
...,...,...
93.0,12,0.0
94.0,12,0.0
95.0,12,0.0


In [62]:
# Convertir la variable en numérique
panel_solde_df["AGE"] = pd.to_numeric(panel_solde_df["AGE"], errors='coerce')  
panel_solde_df["AGE"].describe().round(0)

count    15627752.0
mean           42.0
std             9.0
min             3.0
25%            35.0
50%            41.0
75%            49.0
max           120.0
Name: AGE, dtype: float64

In [78]:
#################################################################
#                               AGE VALIDE                      #
#################################################################

In [63]:
# L'âge de la retraite est fixé à 60 ans. Je vais supposer comme non réponse tous les fonctionnaires de plus de 70 ans et moins de 18 ans
panel_solde_df["AGE_VALIDE"] = panel_solde_df["AGE"].where((panel_solde_df["AGE"] >= 18) & (panel_solde_df["AGE"] <= 70))
panel_solde_df["AGE_VALIDE"].describe().round(0)

count    15625154.0
mean           42.0
std             9.0
min            18.0
25%            35.0
50%            41.0
75%            49.0
max            70.0
Name: AGE_VALIDE, dtype: float64

In [65]:
# Nombre de non réponses au niveau de l'âge
# Calcul des statistiques de AGE_VALIDE
desc_age = panel_solde_df["AGE_VALIDE"].describe().round(0)

# Nombre de non-réponses (âge invalide : <18 ou >70 ou NaN de base)
nb_nan = panel_solde_df["AGE_VALIDE"].isna().sum()

print(desc_age)

count    15625154.0
mean           42.0
std             9.0
min            18.0
25%            35.0
50%            41.0
75%            49.0
max            70.0
Name: AGE_VALIDE, dtype: float64


In [66]:
# Tabulation de la variable Age valide
tab_age_valide = panel_solde_df["AGE_VALIDE"].value_counts(dropna=False).sort_index()
tab_age_valide_pct = panel_solde_df["AGE_VALIDE"].value_counts(normalize=True, dropna=False).sort_index()

pd.DataFrame({
    "Effectif": tab_age_valide,
    "Pourcentage (%)": (tab_age_valide_pct * 100).round(2)
})

,Effectif,Pourcentage (%)
AGE_VALIDE,,
18.0,70,0.00
19.0,327,0.00
20.0,1298,0.01
21.0,3714,0.02
22.0,8329,0.05
23.0,16249,0.10
24.0,31180,0.20
25.0,55275,0.35
26.0,91312,0.58


In [103]:
###################################################################
#                              SEXE                               #                          
###################################################################

In [87]:
# Examen de la variable sexe
panel_solde_df["SEXE"].describe()


count     15627813
unique           2
top       Masculin
freq      10944644
Name: SEXE, dtype: object

In [89]:
tab_sexe = panel_solde_df["SEXE"].value_counts(dropna=False).sort_index()
tab_sexe_pct = panel_solde_df["SEXE"].value_counts(normalize=True, dropna=False).sort_index()

pd.DataFrame({
    "Effectif": tab_sexe,
    "Pourcentage (%)": (tab_sexe_pct * 100).round(2)
})

,Effectif,Pourcentage (%)
SEXE,,
Féminin,4683169,29.97
Masculin,10944644,70.03
NaN,150,0.00


In [91]:
#### Proposition d'imputation du sexe manquant par le mode (valeur la plus fréquente) du groupe année/mois de la collecte
# 1. Création d'une variable SEXE_VALIDE identique à SEXE au départ
panel_solde_df["SEXE_VALIDE"] = panel_solde_df["SEXE"]

# 2. Calcul de la médiane d’âge par ANNEE_COLLECTE et MOIS_COLLECTE
mode_sexe_groupes = (
    panel_solde_df
    .dropna(subset=["SEXE"])  # On ignore les NaN pour calculer le mode
    .groupby(["ANNEE_COLLECTE", "MOIS_COLLECTE"])["SEXE"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
)

# 3. Fonction d’imputation 
def imputer_sexe_par_mode(row):
    if pd.isna(row["SEXE_VALIDE"]):
        return mode_sexe_groupes.get((row["ANNEE_COLLECTE"], row["MOIS_COLLECTE"]), np.nan)
    else:
        return row["SEXE_VALIDE"]

# 4. Nouvelle variable avec l’âge imputé
panel_solde_df["SEXE_VALIDE"] = panel_solde_df.apply(imputer_sexe_par_mode, axis=1)

In [92]:
tab_sexe = panel_solde_df["SEXE_VALIDE"].value_counts(dropna=False).sort_index()
tab_sexe_pct = panel_solde_df["SEXE_VALIDE"].value_counts(normalize=True, dropna=False).sort_index()

pd.DataFrame({
    "Effectif": tab_sexe,
    "Pourcentage (%)": (tab_sexe_pct * 100).round(2)
})

,Effectif,Pourcentage (%)
SEXE_VALIDE,,
Féminin,4683169,29.97
Masculin,10944794,70.03


In [96]:
pd.crosstab(
    index= panel_solde_df["SEXE"],
    columns=panel_solde_df["SEXE_VALIDE"],
    dropna=False,
    margins=True,  # Ajouter une ligne/colonne Total
    margins_name="Total"
)

SEXE_VALIDE,Féminin,Masculin,Total
SEXE,,,
Féminin,4683169,0,4683169.0
Masculin,0,10944644,10944644.0
NaN,0,150,NaN
Total,4683169,10944794,15627963.0


In [83]:
###################################################################
#                           AGE IMPUTE                            #                          
###################################################################

In [100]:
#### Proposition d'imputation de l'âge par l'âge médian de l'année, du mois de collecte et le sexe
# 1. Calcul de la médiane d’âge par ANNEE_COLLECTE, MOIS_COLLECTE et SEXE
mediane_groupes = panel_solde_df.groupby(["ANNEE_COLLECTE", "MOIS_COLLECTE", "SEXE_VALIDE"])["AGE_VALIDE"].median()

# 2. Fonction d’imputation par la médiane du groupe
def imputer_par_mediane(row):
    if pd.isna(row["AGE_VALIDE"]):
        return mediane_groupes.get((row["ANNEE_COLLECTE"], row["MOIS_COLLECTE"], row["SEXE_VALIDE"]), np.nan)
    else:
        return row["AGE_VALIDE"]

# 3. Nouvelle variable avec l’âge imputé
panel_solde_df["AGE_IMPUTE"] = panel_solde_df.apply(imputer_par_mediane, axis=1)

In [101]:
# Tabulation de la variable Age imputé
tab_age_impute = panel_solde_df["AGE_IMPUTE"].value_counts(dropna=False).sort_index()
tab_age_impute_pct = panel_solde_df["AGE_IMPUTE"].value_counts(normalize=True, dropna=False).sort_index()

pd.DataFrame({
    "Effectif": tab_age_impute,
    "Pourcentage (%)": (tab_age_impute_pct * 100).round(2)
})

,Effectif,Pourcentage (%)
AGE_IMPUTE,,
18.0,70,0.00
19.0,327,0.00
20.0,1298,0.01
21.0,3714,0.02
22.0,8329,0.05
23.0,16249,0.10
24.0,31180,0.20
25.0,55275,0.35
26.0,91312,0.58


In [102]:
# Filtrer les individus avec AGE_VALIDE manquant
df_age_nan = panel_solde_df[panel_solde_df["AGE_VALIDE"].isna()]

# Créer la table croisée uniquement sur ce sous-ensemble
pd.crosstab(
    index=df_age_nan["AGE_VALIDE"],
    columns=df_age_nan["AGE_IMPUTE"],
    dropna=False
)

AGE_IMPUTE,39.0,40.0,41.0,42.0
AGE_VALIDE,,,,
NaN,226,12,1062,1509


In [ ]:
###################################################################
#                      SITUATION MATRIMONIALE                     #                          
###################################################################

In [104]:
sit_mat = panel_solde_df["SITUATION MATRIMONIALE"].value_counts(dropna=False).sort_index()
sit_mat_pct = panel_solde_df["SITUATION MATRIMONIALE"].value_counts(normalize=True, dropna=False).sort_index()

pd.DataFrame({
    "Effectif": sit_mat,
    "Pourcentage (%)": (sit_mat_pct * 100).round(2)
})


,Effectif,Pourcentage (%)
SITUATION MATRIMONIALE,,
Cas Particulier,25466,0.16
Célibataire,10021194,64.12
Divorcé Séparé,12637,0.08
Femme Mariée,1415108,9.05
Invalide Marié,99,0.00
Marié,4147361,26.54
Veuf / veuve,6098,0.04


In [106]:
# Dictionnaire de recodage
recodage_matrimoniale = {
    "Célibataire": "Célibataire",
    "Cas Particulier": "Autres",
    "Divorcé Séparé": "Divorcé/Séparé",
    "Femme Mariée": "Marié(e)",
    "Invalide Marié": "Marié(e)",
    "Marié": "Marié(e)",
    "Veuf / veuve": "Veuf/Veuve"
}

# Appliquer le recodage
panel_solde_df["SITUATION_MATRIMONIALE_RECODE"] = panel_solde_df["SITUATION MATRIMONIALE"].replace(recodage_matrimoniale)

In [108]:
sit_mat = panel_solde_df["SITUATION_MATRIMONIALE_RECODE"].value_counts(dropna=False).sort_index()
sit_mat_pct = panel_solde_df["SITUATION_MATRIMONIALE_RECODE"].value_counts(normalize=True, dropna=False).sort_index()

pd.DataFrame({
    "Effectif": sit_mat,
    "Pourcentage (%)": (sit_mat_pct * 100).round(2)
})

,Effectif,Pourcentage (%)
SITUATION_MATRIMONIALE_RECODE,,
Autres,25466,0.16
Célibataire,10021194,64.12
Divorcé/Séparé,12637,0.08
Marié(e),5562568,35.59
Veuf/Veuve,6098,0.04


In [ ]:
###################################################################
#                           NOMBRE D'ENFANTS                      #                          
###################################################################

In [114]:
tab_nbre_enft = panel_solde_df["NOMBRE ENFANT"].value_counts(dropna=False).sort_index()
tab_nbre_enft_pct = panel_solde_df["NOMBRE ENFANT"].value_counts(normalize=True, dropna=False).sort_index()

pd.DataFrame({
    "Effectif": tab_nbre_enft,
    "Pourcentage (%)": (tab_nbre_enft_pct * 100).round(2)
})

,Effectif,Pourcentage (%)
NOMBRE ENFANT,,
0.0,5578049,35.69
1.0,2369618,15.16
2.0,2595972,16.61
3.0,2164280,13.85
4.0,1426419,9.13
5.0,780665,5.00
6.0,476700,3.05
7.0,133859,0.86
8.0,50576,0.32


In [115]:
# Copier la variable d'origine
panel_solde_df["Nbre_enfts_impute"] = panel_solde_df["NOMBRE ENFANT"]

# Remplacer les NaN par 0
panel_solde_df["Nbre_enfts_impute"] = panel_solde_df["Nbre_enfts_apure"].fillna(0)

In [116]:
tab_nbre_enft = panel_solde_df["Nbre_enfts_impute"].value_counts(dropna=False).sort_index()
tab_nbre_enft_pct = panel_solde_df["Nbre_enfts_impute"].value_counts(normalize=True, dropna=False).sort_index()

pd.DataFrame({
    "Effectif": tab_nbre_enft,
    "Pourcentage (%)": (tab_nbre_enft_pct * 100).round(2)
})

,Effectif,Pourcentage (%)
Nbre_enfts_impute,,
0.0,5604678,35.86
1.0,2369618,15.16
2.0,2595972,16.61
3.0,2164280,13.85
4.0,1426419,9.13
5.0,780665,5.00
6.0,476700,3.05
7.0,133859,0.86
8.0,50576,0.32


In [ ]:
###################################################################
#                         LIEU D'AFFECTATION                      #                          
###################################################################

In [ ]:
# Prendre la liste des localités du recensement et faire le merge pour avoir les différentes entités administratives

In [ ]:
###################################################################
#                              SERVICE                            #                          
###################################################################

In [123]:
# Nous cherchons à vérifier que r vérifier que chaque organisme est associé un seul code organisme
# Regrouper par ORGANISME et compter le nombre de codes distincts
verif_unicite = panel_solde_df.groupby("ORGANISME")["CODE_ORGANISME"].nunique()

# Afficher uniquement les cas problématiques (plus d’un code par organisme)
cas_non_uniques = verif_unicite[verif_unicite > 1]

# Vérifier s'il y a des cas non conformes
if cas_non_uniques.empty:
    print("OK : chaque organisme est associé à un seul CODE_ORGANISME.")
else:
    print("Attention : certains organismes sont associés à plusieurs codes.")

    

OK : chaque organisme est associé à un seul CODE_ORGANISME.


In [ ]:
###################################################################
#                              EMPLOIS                            #                          
###################################################################

In [ ]:
## J'ai un fichier des codes de tous les emplois que je souhaites associer au libellés qui est dans ma base de travail

In [119]:
print (panel_solde_df.columns)

Index(['MATRICULE||CODE_ORGANISME', 'MATRICULE', 'NOM', 'DATE NAISSANCE',
       'SEXE', 'SITUATION MATRIMONIALE', 'NOMBRE ENFANT', 'LIEU AFFECTATION',
       'SERVICE', 'ORGANISME', 'PRISE DE SERVICE', 'SITUATION',
       'DATE DEBUT SITUATION', 'EMPLOI', 'FONCTION', 'GRADE', 'CLASSE/ECHELON',
       'MONTANT BRUT', 'MONTANT NET', 'RETENUE  PENSION', 'IMPOT ',
       'CHARGE PATRONALE', 'MOIS/ANNEE', 'GRADE 2', 'AGE RETRAITE',
       'DATE RETRAITE', 'PERIODE', 'DATE_COLLECTE', 'CODE_ORGANISME',
       'ANNEE_NAISSANCE', 'MOIS_NAISSANCE', 'JOUR_NAISSANCE', 'ANNEE_COLLECTE',
       'MOIS_COLLECTE', 'JOUR_COLLECTE', 'individus_naissances_diff',
       'nbre_date_naissances', 'DATE_NAISSANCE_CORRIGEE',
       'ANNEE_NAISSANCE_CORRIGEE', 'MOIS_NAISSANCE_CORRIGEE',
       'JOUR_NAISSANCE_CORRIGEE', 'AGE', 'AGE_VALIDE', 'AGE_IMPUTE',
       'SEXE_VALIDE', 'SITUATION_MATRIMONIALE_RECODE', 'Nbre_enfts_apure',
       'Nbre_enfts_impute'],
      dtype='object')


In [ ]:
###################################################################
#                              ORGANISME                           #                          
###################################################################

In [60]:
tab_date_retraite = panel_solde_df["DATE RETRAITE"].value_counts(dropna=False).sort_index()
tab_date_retraite_pct = panel_solde_df["DATE RETRAITE"].value_counts(normalize=True, dropna=False).sort_index()

pd.DataFrame({
    "Effectif": tab_date_retraite,
    "Pourcentage (%)": (tab_date_retraite_pct * 100).round(2)
})

,Effectif,Pourcentage (%)
DATE RETRAITE,,
NaN,15627963,100.0


In [ ]:
tab_nbre_enft = panel_solde_df["NOMBRE ENFANT"].value_counts(dropna=False).sort_index()
tab_nbre_enft_pct = panel_solde_df["NOMBRE ENFANT"].value_counts(normalize=True, dropna=False).sort_index()

pd.DataFrame({
    "Effectif": tab_nbre_enft,
    "Pourcentage (%)": (tab_nbre_enft_pct * 100).round(2)
})

In [61]:
tab_age_retraite = panel_solde_df["AGE RETRAITE"].value_counts(dropna=False).sort_index()
tab_age_retraite_pct = panel_solde_df["AGE RETRAITE"].value_counts(normalize=True, dropna=False).sort_index()

pd.DataFrame({
    "Effectif": tab_age_retraite,
    "Pourcentage (%)": (tab_age_retraite_pct * 100).round(2)
})

,Effectif,Pourcentage (%)
AGE RETRAITE,,
NaN,15627963,100.0


In [65]:
tab_lieu_aff = panel_solde_df["LIEU AFFECTATION"].value_counts(dropna=False).sort_index()
tab_lieu_aff_pct = panel_solde_df["LIEU AFFECTATION"].value_counts(normalize=True, dropna=False).sort_index()

pd.DataFrame({
    "Effectif": tab_lieu_aff,
    "Pourcentage (%)": (tab_lieu_aff_pct * 100).round(2)
})

,Effectif,Pourcentage (%)
LIEU AFFECTATION,,
A Réaffecter hors Korhogo,1727,0.01
AHEREMOU 2,120,0.00
ALLANGOUASSOU,72,0.00
ANCIEN PROZI,162,0.00
ANGOUA KOUADIOKRO,72,0.00
...,...,...
Zro,74,0.00
Zuénoula,58384,0.37
Zéaglo,1009,0.01


In [ ]:
LIEU AFFECTATION
SITUATION

In [66]:
tab_sit = panel_solde_df["SITUATION"].value_counts(dropna=False).sort_index()
tab_sit_pct = panel_solde_df["SITUATION"].value_counts(normalize=True, dropna=False).sort_index()

pd.DataFrame({
    "Effectif": tab_sit,
    "Pourcentage (%)": (tab_sit_pct * 100).round(2)
})

,Effectif,Pourcentage (%)
SITUATION,,
Abandon de Poste,9301,0.06
Agent en Liquidation de Droits,66,0.00
Annulé,10167,0.07
Arret FP Situation Irrégulière,1,0.00
Arret Recensement 2011,753,0.00
Arrêt // Grève,3807,0.02
Arrêt CS Santé,23,0.00
Arrêt Corps Habilles,281,0.00
Arrêt FP Absence Notation,4778,0.03


In [71]:
# Calcul du salaire moyen par situation
salaire_moyen_par_situation = panel_solde_df.groupby("SITUATION")["MONTANT BRUT"].mean().round(2)

# Créer un tableau combiné avec effectif, pourcentage et salaire moyen
tableau_final = pd.DataFrame({
    "Effectif": tab_sit,
    "Pourcentage (%)": (tab_sit_pct * 100).round(2),
    "Salaire moyen": salaire_moyen_par_situation
})

tableau_final

,Effectif,Pourcentage (%),Salaire moyen
SITUATION,,,
Abandon de Poste,9301,0.06,391167.43
Agent en Liquidation de Droits,66,0.00,2702157.18
Annulé,10167,0.07,660701.08
Arret FP Situation Irrégulière,1,0.00,50400.00
Arret Recensement 2011,753,0.00,1644154.70
Arrêt // Grève,3807,0.02,296146.00
Arrêt CS Santé,23,0.00,334382.17
Arrêt Corps Habilles,281,0.00,608525.24
Arrêt FP Absence Notation,4778,0.03,379191.61


In [69]:
panel_solde_df["MONTANT BRUT"].describe().round()

count    15627963.0
mean       465586.0
std        546440.0
min           542.0
25%        274176.0
50%        375040.0
75%        514147.0
max      76801094.0
Name: MONTANT BRUT, dtype: float64

In [68]:
tab_mont_brut = panel_solde_df["MONTANT BRUT"].value_counts(dropna=False).sort_index()
tab_mont_brut_pct = panel_solde_df["MONTANT BRUT"].value_counts(normalize=True, dropna=False).sort_index()

pd.DataFrame({
    "Effectif": tab_mont_brut ,
    "Pourcentage (%)": (tab_mont_brut_pct * 100).round(2)
})

,Effectif,Pourcentage (%)
MONTANT BRUT,,
542,1,0.0
1948,1,0.0
2164,1,0.0
2303,1,0.0
2450,1,0.0
...,...,...
46656723,1,0.0
46888230,3,0.0
52375865,1,0.0
